# Lecture 15
## Combining and reshaping tables
## Week 6 Monday
## Miles Chen, PhD

**How do I organize data for analysis and plotting?**

Adapted from *Python Data Science Handbook* by Jake VanderPlas and
*Python for Data Analysis* by Wes McKinney.

# Where we are going

In lecture 13, we selected observations and created variables. In lecture 14,
we cleaned records and summarized groups, keeping track of denominators.

Today we will:

- Combine information from two tables using a **key** and a **left join**.
- Check that a join has the intended matches and number of rows.
- Move between wide and long formats using `melt` and `pivot`.
- Distinguish reshaping existing values from aggregating several observations.

Keep asking: **What does one row represent, and what identifies it?**

In [1]:
import pandas as pd

titanic = pd.read_csv("../data/titanic.csv")
print("pandas version:", pd.__version__)
print("Rows and columns:", titanic.shape)
titanic.head()

pandas version: 3.0.1
Rows and columns: (891, 12)


,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Start with a familiar summary

The passenger table has one row per passenger. The summary below has one row
per **passenger-class and sex combination**.

`size` counts rows; `count` counts nonmissing values. The mean of a recorded
0/1 outcome is the proportion of recorded outcomes equal to 1.

In [2]:
group_summary = (
    titanic.groupby(["pclass", "sex"], as_index=False, dropna=False)
    .agg(
        n_passengers=("passengerid", "size"),
        n_survival=("survived", "count"),
        survival_rate=("survived", "mean"),
        n_age=("age", "count"),
        mean_age=("age", "mean"),
    )
    .sort_values(["pclass", "sex"])
)
group_summary

,pclass,sex,n_passengers,n_survival,survival_rate,n_age,mean_age
0,1,female,94,94,0.968085,85,34.611765
1,1,male,122,122,0.368852,101,41.281386
2,2,female,76,76,0.921053,74,28.722973
3,2,male,108,108,0.157407,99,30.740707
4,3,female,144,144,0.500000,102,21.750000
5,3,male,347,347,0.135447,253,26.507589


# Keys identify records

A **key** is a column, or a combination of columns, that identifies a row in
a particular table. A usable identifier should be unique and nonmissing.

| Table | One row represents | Candidate key |
| --- | --- | --- |
| `titanic` | A passenger | `passengerid` |
| `group_summary` | A class/sex group | `pclass` and `sex` together |

A column's role depends on the table: `pclass` alone is not unique in either
of these tables. A ticket may cover several passengers, so it is not a
passenger identifier.

In [3]:
print("Passenger IDs are unique:", titanic["passengerid"].is_unique)
print("Missing passenger IDs:", titanic["passengerid"].isna().sum())
print("Repeated class/sex keys:", group_summary.duplicated(["pclass", "sex"]).sum())

Passenger IDs are unique: True
Missing passenger IDs: 0
Repeated class/sex keys: 0


# Add information from a lookup table

The Titanic data records the port of embarkation as a short code. We have a
separate table that gives each code's name.

Several passengers can have the same port code, but each code should have
just **one** name in the lookup table. This is a **many-to-one** relationship.

In `dplyr`, this operation is a `left_join()`.

In [4]:
ports = pd.DataFrame({
    "embarked": ["C", "Q", "S"],
    "port_name": ["Cherbourg", "Queenstown", "Southampton"],
})
ports

,embarked,port_name
0,C,Cherbourg
1,Q,Queenstown
2,S,Southampton


In [5]:
print("Lookup keys are unique:", ports["embarked"].is_unique)
print("Missing lookup keys:", ports["embarked"].isna().sum())
titanic[["passengerid", "embarked"]].head()

Lookup keys are unique: True
Missing lookup keys: 0


,passengerid,embarked
0,1,S
1,2,C
2,3,S
3,4,S
4,5,S


# A left join matches values

`left.merge(right, on=..., how="left")` keeps every row from the left table
and attaches matching information from the right table.

- `on="embarked"` identifies the column used to match records.
- `validate="many_to_one"` checks that each right-side key occurs at most once.
- `indicator=True` adds `_merge`, which records whether a match was found.

A left join preserves the original row count **when each left row has at
most one match**.

In [6]:
passengers_with_ports = titanic.merge(
    ports,
    on="embarked",
    how="left",
    validate="many_to_one",
    indicator=True,
)
passengers_with_ports[["passengerid", "embarked", "port_name", "_merge"]].head()

,passengerid,embarked,port_name,_merge
0,1,S,Southampton,both
1,2,C,Cherbourg,both
2,3,S,Southampton,both
3,4,S,Southampton,both
4,5,S,Southampton,both


# Check the result before using it

We expect 891 passenger rows, unique passenger IDs, and no lost passengers.

In [7]:
print("Before join:", len(titanic))
print("After join:", len(passengers_with_ports))
print("IDs still unique:", passengers_with_ports["passengerid"].is_unique)
passengers_with_ports["_merge"].value_counts()

Before join: 891
After join: 891
IDs still unique: True


_merge
both          889
left_only       2
right_only      0
Name: count, dtype: int64

In [8]:
unmatched = passengers_with_ports.loc[
    passengers_with_ports["_merge"] == "left_only",
    ["passengerid", "name", "embarked", "port_name"],
]
unmatched

,passengerid,name,embarked,port_name
61,62,"Icard, Miss. Amelie",NaN,NaN
829,830,"Stone, Mrs. George Nelson (Martha Evelyn)",NaN,NaN


# An unmatched record is information

Two passengers have no recorded embarkation code. The left join retains
them, with a missing `port_name`. A misspelled or unexpected code could also
produce an unmatched row, so inspect the actual records.

An **inner join** retains only matched rows. That would exclude these two
passengers and change the population available for later summaries.

Neither join type is always correct; choose based on the question you want
the resulting table to answer.

In [9]:
matched_passengers = titanic.merge(
    ports, on="embarked", how="inner", validate="many_to_one"
)
print("Left join rows:", len(passengers_with_ports))
print("Inner join rows:", len(matched_passengers))

Left join rows: 891
Inner join rows: 889


# What if the lookup table has a repeated key?

Suppose two entries for Southampton were accidentally included. Each
passenger with code `S` would match **both** entries.

Predict what happens to the row count before running the next cell. Could
a survival summary now give some passengers more weight than others?

In [10]:
ports_bad = pd.DataFrame({
    "embarked": ["C", "Q", "S", "S"],
    "port_name": ["Cherbourg", "Queenstown", "Southampton", "Southampton, UK"],
})
expanded = titanic.merge(ports_bad, on="embarked", how="left")
print("Original rows:", len(titanic))
print("Rows after the unchecked join:", len(expanded))
ports_bad.loc[ports_bad.duplicated("embarked", keep=False)]

Original rows: 891
Rows after the unchecked join: 1535


,embarked,port_name
2,S,Southampton
3,S,"Southampton, UK"


In [11]:
# This error is intentional: validation catches the repeated lookup key.
try:
    titanic.merge(
        ports_bad, on="embarked", how="left", validate="many_to_one"
    )
except pd.errors.MergeError as error:
    print("Join stopped:", error)

Join stopped: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 embarked
       S ...


# Resolve the meaning of the duplicate

There are 644 passengers with code `S`, so the unchecked join adds 644 extra
rows: 891 becomes 1,535. Those passengers would be counted twice.

Investigate why a key repeats. A mistaken duplicate, conflicting lookup
information, and a genuine one-to-many relationship require different
decisions. Arbitrarily keeping the first row can hide the problem.

Here, `ports` is our intended one-row-per-port lookup table.

# Your turn: label the class summary

Use the lookup table below to add `class_name` to `group_summary`.

1. Choose the join key and the expected relationship.
2. Keep all class/sex groups and check the result's row count.
3. Check for missing class names.
4. Explain why the output still has six rows even though the lookup has three.

In [12]:
class_labels = pd.DataFrame({
    "pclass": [1, 2, 3],
    "class_name": ["First", "Second", "Third"],
})
class_labels

,pclass,class_name
0,1,First
1,2,Second
2,3,Third


In [13]:
# Write your join and checks here.

# One solution

The left table has two sex groups per class. Each row matches one class
label, so the six group rows remain six group rows.

In [14]:
analysis_table = group_summary.merge(
    class_labels, on="pclass", how="left", validate="many_to_one"
).sort_values(["pclass", "sex"])

print("Rows before and after:", len(group_summary), len(analysis_table))
print("Missing class names:", analysis_table["class_name"].isna().sum())
analysis_table

Rows before and after: 6 6
Missing class names: 0


,pclass,sex,n_passengers,n_survival,survival_rate,n_age,mean_age,class_name
0,1,female,94,94,0.968085,85,34.611765,First
1,1,male,122,122,0.368852,101,41.281386,First
2,2,female,76,76,0.921053,74,28.722973,Second
3,2,male,108,108,0.157407,99,30.740707,Second
4,3,female,144,144,0.500000,102,21.750000,Third
5,3,male,347,347,0.135447,253,26.507589,Third


# Wide and long describe the arrangement of values

In this fictional quiz table, one row represents one student. Scores for
different quizzes occupy different columns: this is a **wide** arrangement.

For comparing quizzes, we may want one row per student/quiz combination,
with a column for the quiz name and a column for the score: a **long**
arrangement.

Reshaping changes the arrangement. It does not have to calculate new scores.

In [15]:
scores_wide = pd.DataFrame({
    "student": ["Ada", "Ben"],
    "quiz1": [8.0, 6.0],
    "quiz2": [9.0, None],
})
scores_wide

,student,quiz1,quiz2
0,Ada,8.0,9.0
1,Ben,6.0,NaN


# `melt`: wide to long

- `id_vars`: columns that identify the original rows and stay as columns.
- `value_vars`: columns whose values will be gathered into one column.
- `var_name`: name for the column containing the old column labels.
- `value_name`: name for the column containing their values.

This is similar to `tidyr::pivot_longer()`.

**Predict:** How many rows will the long table have? What happens to Ben's
missing second quiz score?

In [16]:
scores_long = scores_wide.melt(
    id_vars="student",
    value_vars=["quiz1", "quiz2"],
    var_name="quiz",
    value_name="score",
)
scores_long

,student,quiz,score
0,Ada,quiz1,8.0
1,Ben,quiz1,6.0
2,Ada,quiz2,9.0
3,Ben,quiz2,NaN


In [17]:
print("Rows:", len(scores_long))
print("Recorded scores:", scores_long["score"].count())
print("Repeated student/quiz keys:", scores_long.duplicated(["student", "quiz"]).sum())

Rows: 4
Recorded scores: 3
Repeated student/quiz keys: 0


# The missing measurement is still present

Two students times two quizzes gives four rows, but only three recorded
scores. `melt` keeps the missing score as a row with a missing value.

Dropping that row would be a separate cleaning decision. Remember the
distinction from lecture 14: the number of rows need not be the number of
observed measurements.

We gathered quiz scores because they measure the same kind of quantity.
Combining age and survival rate into a generic `value` column would require
much more care about units and interpretation.

# `pivot`: long to wide, without aggregation

Supply the identifiers for the output rows, the values that become column
labels, and the values that fill the table.

Each student/quiz combination must have **at most one row** in the input.
`pivot` cannot choose between several scores for the same output position.

This is similar to `tidyr::pivot_wider()` when each cell has a single value.

In [18]:
scores_back = scores_long.pivot(
    index="student", columns="quiz", values="score"
)
scores_back

quiz,quiz1,quiz2
student,,
Ada,8.0,9.0
Ben,6.0,NaN


# Keep identifiers as ordinary columns when useful

`pivot` puts the row identifier in the index. `reset_index()` brings it back
into an ordinary column, which is often convenient for later joins and plots.

The word `quiz` above the column labels is the **name of the column axis**;
it is not an additional data column. `rename_axis(columns=None)` removes
that display label.

In [19]:
scores_back = scores_back.reset_index().rename_axis(columns=None)
scores_back

,student,quiz1,quiz2
0,Ada,8.0,9.0
1,Ben,6.0,NaN


# Reshaping requires an unambiguous destination

The next table contains **sales transactions**. Two separate transactions
involved apples on January 1. They are legitimate observations, not an
accidental duplicate record.

If we want one row per date and one column per fruit, two values are trying
to occupy the January 1/apple cell. What should that cell contain?

In [20]:
sales = pd.DataFrame({
    "date": ["Jan 1", "Jan 1", "Jan 1", "Jan 2"],
    "fruit": ["Apple", "Apple", "Banana", "Apple"],
    "sold": [5, 3, 8, 6],
})
sales

,date,fruit,sold
0,Jan 1,Apple,5
1,Jan 1,Apple,3
2,Jan 1,Banana,8
3,Jan 2,Apple,6


In [21]:
sales.loc[sales.duplicated(["date", "fruit"], keep=False)]

,date,fruit,sold
0,Jan 1,Apple,5
1,Jan 1,Apple,3


In [22]:
# This error is intentional: two transactions share an output position.
try:
    sales.pivot(index="date", columns="fruit", values="sold")
except ValueError as error:
    print("Cannot reshape without a decision:", error)

Cannot reshape without a decision: Index contains duplicate entries, cannot reshape


# `pivot_table`: aggregate and then arrange

If the question is **How many items of each fruit were sold on each date?**,
sum the transactions within each date/fruit group.

Specify `aggfunc` explicitly. The default is a mean, which answers a
different question: the average number sold per recorded transaction.

In [23]:
daily_sales = sales.pivot_table(
    index="date", columns="fruit", values="sold", aggfunc="sum"
)
daily_sales

fruit,Apple,Banana
date,,
Jan 1,8.0,8.0
Jan 2,6.0,NaN


In [24]:
mean_transaction = sales.pivot_table(
    index="date", columns="fruit", values="sold", aggfunc="mean"
)
mean_transaction

fruit,Apple,Banana
date,,
Jan 1,4.0,8.0
Jan 2,6.0,NaN


# Aggregation changes what a value means

For January 1 apples, the total is **8** and the mean transaction size is
**4**. Both are valid calculations, but they answer different questions.

After summing, we cannot recover the original transactions of 5 and 3 from
the total of 8. Reshaping and summarizing are different operations.

January 2 bananas are missing because no such transaction appears in the
input. That is not automatically evidence of zero sales. Use
`fill_value=0` only when the data-collection process justifies interpreting
an absent combination as zero.

# The same result with familiar grouping

`groupby(...).agg(...)` lets us see the aggregation step separately from the
reshaping step. Once each date/fruit combination has one total, `pivot` has
an unambiguous value for every observed combination.

In [25]:
daily_sales_long = sales.groupby(["date", "fruit"], as_index=False).agg(
    total_sold=("sold", "sum")
)
daily_sales_long

,date,fruit,total_sold
0,Jan 1,Apple,8
1,Jan 1,Banana,8
2,Jan 2,Apple,6


In [26]:
daily_sales_long.pivot(index="date", columns="fruit", values="total_sold")

fruit,Apple,Banana
date,,
Jan 1,8.0,8.0
Jan 2,6.0,NaN


# Return to the Titanic comparison

`analysis_table` already has one survival rate per class/sex group. We can
arrange those rates for comparison with `pivot`; no new averaging is needed.

**Before running:** How many rows and rate columns should the result have?

In [27]:
survival_wide = analysis_table.pivot(
    index="pclass", columns="sex", values="survival_rate"
).reset_index().rename_axis(columns=None)
survival_wide

,pclass,female,male
0,1,0.968085,0.368852
1,2,0.921053,0.157407
2,3,0.500000,0.135447


# Put the comparison back into long format

A long table gives plotting tools a variable for each visual role:
`pclass` for an x-axis, `survival_rate` for a y-axis, and `sex` for color or
grouping. The plotting function can refer to these column names.

Specify the two rate columns explicitly so we do not accidentally gather
an identifier or a count as though it were another rate.

In [28]:
survival_long = survival_wide.melt(
    id_vars="pclass",
    value_vars=["female", "male"],
    var_name="sex",
    value_name="survival_rate",
).sort_values(["pclass", "sex"]).reset_index(drop=True)
survival_long

,pclass,sex,survival_rate
0,1,female,0.968085
1,1,male,0.368852
2,2,female,0.921053
3,2,male,0.157407
4,3,female,0.500000
5,3,male,0.135447


# A rate still needs a denominator

The reshaped table contains the rates we selected. It does not automatically
carry along the counts and age summaries from the original table.

Keep `analysis_table` as the fuller analysis result. `n_survival` is the
denominator for each survival rate; `n_age` is the denominator for each
mean age. Here all survival outcomes are recorded, but some ages are missing.

The original group summary was already long enough for many plots. Reshape
because a task needs a different arrangement, not as a mandatory step.

# Your turn: prepare an analysis-ready table

Starting with `analysis_table`:

1. Create `plot_table` with `pclass`, `class_name`, `sex`, `n_passengers`,
   `n_survival`, and `survival_rate`. Sort by class and sex.
2. Check that there are six distinct class/sex groups and that their
   passenger counts sum to the number of passengers in the source table.
3. Identify the denominator for the third-class female survival rate.
4. Explain why a simple average of the six group rates need not equal the
   overall passenger survival rate.

In [29]:
# Create plot_table and check its rows, keys, counts, and interpretation.

# One solution

A row now represents a class/sex group, with its label, size, denominator,
and survival rate all available in ordinary columns.

In [30]:
plot_table = analysis_table[
    ["pclass", "class_name", "sex", "n_passengers", "n_survival", "survival_rate"]
].sort_values(["pclass", "sex"]).reset_index(drop=True)

print("Group rows:", len(plot_table))
print("Repeated class/sex keys:", plot_table.duplicated(["pclass", "sex"]).sum())
print("Passengers represented:", plot_table["n_passengers"].sum())
plot_table

Group rows: 6
Repeated class/sex keys: 0
Passengers represented: 891


,pclass,class_name,sex,n_passengers,n_survival,survival_rate
0,1,First,female,94,94,0.968085
1,1,First,male,122,122,0.368852
2,2,Second,female,76,76,0.921053
3,2,Second,male,108,108,0.157407
4,3,Third,female,144,144,0.500000
5,3,Third,male,347,347,0.135447


In [31]:
plot_table.loc[(plot_table["pclass"] == 3) & (plot_table["sex"] == "female")]

,pclass,class_name,sex,n_passengers,n_survival,survival_rate
4,3,Third,female,144,144,0.5


# Explain the comparison

The third-class female group contains 144 passengers with recorded survival
outcomes. Its rate of 0.50 means 72 of those 144 passengers survived.

An unweighted mean of the six group rates gives equal weight to groups of
different sizes. The overall passenger rate gives each recorded passenger
outcome equal weight. We should use the relevant denominators when
combining group rates.

These are descriptive comparisons in this dataset. The tables alone do
not establish the causes of the differences.

# Choose the operation from the question

| Question | Operation | Check |
| --- | --- | --- |
| Can I add information about each key? | `merge` | Key relationship, row count, unmatched rows |
| Can these measurement columns become rows? | `melt` | Identifiers, units, missing values |
| Can these rows become columns? | `pivot` | Unique row/column combinations |
| Do several observations need one summary value? | `groupby` or `pivot_table` | Aggregation rule and denominator |

Next time, we will express these familiar table operations in Polars.
The questions about rows, keys, missing values, and denominators carry over.

# attach a group summary to individual rows

Sometimes we want each passenger row to include its group's survival rate.
The lookup key now has **two columns**: class and sex together.

The group rate describes the group; it is not a new observed passenger
outcome. Attaching it does not turn it into an individual prediction.

In [32]:
passenger_context = titanic.merge(
    group_summary[["pclass", "sex", "survival_rate"]],
    on=["pclass", "sex"],
    how="left",
    validate="many_to_one",
)
print("Passenger rows:", len(passenger_context))
passenger_context[["passengerid", "pclass", "sex", "survived", "survival_rate"]].head()

Passenger rows: 891


,passengerid,pclass,sex,survived,survival_rate
0,1,3,male,0,0.135447
1,2,1,female,1,0.968085
2,3,3,female,1,0.500000
3,4,1,female,1,0.968085
4,5,3,male,0,0.135447


# missing join keys need a policy

In pandas, missing keys on the left can match missing keys on the right.
This differs from ordinary SQL joins and the default behavior in Polars.

Our port lookup has no missing keys, so the two passengers with missing
embarkation codes stayed unmatched. Check missing keys in **both** tables
when deciding what a match should mean.

# References

- [pandas: merge and join](https://pandas.pydata.org/docs/user_guide/merging.html)
- [pandas: reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)
- [`DataFrame.merge`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)
- [`DataFrame.melt`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html)
- [`DataFrame.pivot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html)
- [`DataFrame.pivot_table`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html)